In [16]:
import numpy as np
import pandas as pd

In [17]:
# Data preparation

dataset = pd.read_csv('train.csv')
X_train = dataset.drop(columns=['Target'])
y_train = dataset['Target']

ones = np.ones((X_train.shape[0], 1))
X_train = np.column_stack([ones, X_train.values])

In [18]:
x0 = np.array([-1.47, 0, 0, 0, 0])

def f_sum(x, U, v):
    dots = U @ x
    return np.sum((1 - v) * dots + np.log(1 + np.exp(-dots)))

def grad_f_eval(x, U, v):
    g = 1/(1+np.exp(-U@x))
    return U.T @ (g-v)

def gradient_method_const(f, x0, U, v, c=3e-5, epsilon=1e-3, max_iters=3_000_000):
    x = x0.astype(float).copy()
    k = 0
    x_iter = [x.copy()]
    f_val = [f(x, U, v)]
    while k < max_iters:
        grad = U.T @ (1/(1+np.exp(-U@x)) - v)
        if np.linalg.norm(grad) <= epsilon:
            break
        x -= c * grad
        k += 1
        x_iter.append(x.copy())
        f_val.append(f(x, U, v))

    f_val_check = all(f_val[i] >= f_val[i+1] for i in range(len(f_val)-1))
    print(f"Hodnota ucelovej funkcie nerastie: {f_val_check}")
    print(f"Pocet iteracii: {k}")

    return x, x_iter

import time
for c in [3e-7, 3e-5, 3e-3]:
    print(f"\n--- c = {c} ---")
    start = time.time()
    vysl, _ = gradient_method_const(f_sum, x0, X_train, y_train.values,
                                     c=c)

    elapsed = time.time() - start
    print(f"Výsledok: {vysl}")
    print(f"Čas: {elapsed:.2f}s")


--- c = 3e-07 ---
Hodnota ucelovej funkcie nerastie: True
Pocet iteracii: 3000000
Výsledok: [-0.88390857  2.97133462  3.58735415  4.45600849  0.00750751]
Čas: 73.42s

--- c = 3e-05 ---
Hodnota ucelovej funkcie nerastie: True
Pocet iteracii: 2736859
Výsledok: [-1.47360963e+00  3.07175527e+00  1.01989855e+01  1.76905539e+01
  4.84667713e-03]
Čas: 66.48s

--- c = 0.003 ---
Hodnota ucelovej funkcie nerastie: False
Pocet iteracii: 3000000
Výsledok: [-8.27305581  8.23145812 22.59051432 48.10574825  0.40823205]
Čas: 73.24s
